# ZIP เดียว + Drive: Train/Validation และประเมิน Baseline/A–D ครบใน notebook เดียว
**ใช้ฉบับนี้แทนไฟล์ก่อนหน้า: เปิด Colab GPU แล้ว Run all**

1. อัปโหลด `bpp_4trials_pretrained_start_review(2).zip` เพียงหนึ่งไฟล์ (ชื่อเปลี่ยนได้)
2. พล็อต Train/Validation แยก A–D แล้วรวม จากข้อมูลใน ZIP
3. เชื่อม Drive และโหลด best.pt ของ A–D พร้อม source cache/masks เพื่อรันโมเดลใหม่
4. สร้างภาพ baseline เทียบ A–D ทั้ง 9 คู่ วัดเมทริกใหม่ และพล็อตกราฟ
5. ดาวน์โหลด ZIP เดียวที่รวม PNG, CSV และ PDF ทั้งสองส่วน พร้อมเก็บใน Drive

**ไม่ฝึกใหม่ ไม่ใช้ checkpoint Full150 รุ่นเก่า ไม่ต้องอัปโหลด ZIP ผลลัพธ์สองไฟล์**

สถานะและ best epoch อ่านจาก ZIP ไม่อ่าน manifest.json ใน Drive มาตัดสินความครบถ้วน แต่ยังตรวจ checkpoint ให้ตรง trial/epoch/config และตรวจ hash cache/masks เพื่อป้องกันใช้คนละรอบ

Train curve เป็นค่าระหว่าง updates; validation curve มาจาก log หลังจบ epoch ส่วนการประเมินใหม่เป็นคนละตารางและมีตาราง delta เปรียบเทียบให้ ผลทั้งหมดใช้ validation เดิม ไม่ใช่ test. ไม่มี train IoU จึงไม่สร้างค่าขึ้นเอง


## เพิ่มส่วน TEST แล้ว
หลัง validation จะล็อก best checkpoints A:27, B:24, C:26, D:23 (ตรวจจากรายงานที่อัปโหลด) แล้วประเมิน baseline และ A–D บน test_candidate เดียวกัน ไม่ฝึกและไม่เลือกโมเดลจาก test

เตรียมเพิ่ม: `HairFastGAN_Dataset_Split_v2.zip` (ชุดภาพจริงประมาณ 199 MB ไม่ใช่ review ZIP) ระบุ path ใน Drive ที่ DATASET_ZIP หรืออัปโหลดเมื่อถึงเซลล์ TEST

ชุดเดิมมี 10 ภาพหน้า / 7 กลุ่มบุคคล, เรฟทรง 5 และเรฟสี 5 ไม่มีคู่สำเร็จรูป: โค้ดกำหนด 10 คู่จาก seed 3407 ก่อนประเมิน หน้าละ 1 คู่ เรฟละ 2 ครั้ง บันทึก test_pairs.json

ตรวจ overlap ID/hash/pixels/group กับ train/validation รอบปัจจุบัน การซ้ำกับข้อมูลพัฒนาเก่าทุกชุดยังไม่ได้รับรอง จึงคงชื่อ test_candidate ในรายงาน

เตรียม test ด้วย shape rotation OFF, color rotation ON, mixing .95 และ official SEAN/e4e, masks อัตโนมัติคงที่ร่วมกันทุกโมเดล ต่างจาก validation ที่มี masks ตรวจแล้ว ไม่ตีความความต่าง val/test ว่าเกิดจากการ generalize อย่างเดียว

ผล test อยู่ใน 03_test: ภาพเทียบ 5 โมเดล, เมทริกรายคู่, mean/SD, mean แยกกลุ่มบุคคล, PDF, กราฟ error bars แบบ SD และ checkpoint hashes ไม่มี winner ที่เลือกด้วย test

หากมี runtime เดิมที่รัน validation notebook รุ่นก่อนจบแล้วและตัวแปรยังอยู่ สามารถรันเฉพาะเซลล์ TEST และเซลล์บันทึกสุดท้ายได้ Otherwise run all. ส่วน GPU ต้องตรวจใน Colab; ยังไม่อ้างว่าทดสอบ inference สำเร็จที่นี่


In [ ]:
%pip -q install numpy pandas matplotlib


In [ ]:
from google.colab import files
from pathlib import Path
import json, zipfile, uuid, pandas as pd
uploaded=files.upload()  # เลือก review ZIP หนึ่งไฟล์เท่านั้น
if len(uploaded)!=1:raise ValueError('เลือก review ZIP เพียงหนึ่งไฟล์')
REVIEW_ZIP=Path(next(iter(uploaded))).resolve()
if not zipfile.is_zipfile(REVIEW_ZIP):raise ValueError('ไม่ใช่ ZIP ที่อ่านได้')
LOCAL_REPORT=Path('/content')/('ABCD_complete_report_'+uuid.uuid4().hex[:10])
LOCAL_REPORT.mkdir(exist_ok=False)
def zip_member(z,name):
    matches=[n for n in z.namelist() if n==name or n.endswith('/'+name)]
    if len(matches)!=1:raise ValueError('ZIP ต้องมีไฟล์เดียวสำหรับ '+name)
    return matches[0]
with zipfile.ZipFile(REVIEW_ZIP) as z:
    ZIP_MANIFESTS={t:json.loads(z.read(zip_member(z,f'trial_{t}/manifest.json'))) for t in 'ABCD'}
    ZIP_BOARD=pd.read_csv(z.open(zip_member(z,'leaderboard.csv'))).set_index('trial')
    for t,m in ZIP_MANIFESTS.items():
        tr=pd.read_csv(z.open(zip_member(z,f'trial_{t}/training_steps.csv')))
        v=pd.read_csv(z.open(zip_member(z,f'trial_{t}/metrics.csv')))
        v=v[(v.epoch>=0)&(v.split=='validation')]
        tc=tr.groupby('epoch').size();vc=v.groupby('epoch')['case'].nunique()
        assert tc.index.tolist()==list(range(1,31)) and (tc==150).all() and tr['update'].nunique()==4500, f'{t}: training log incomplete'
        assert vc.index.tolist()==list(range(31)) and (vc==9).all() and not v.duplicated(['epoch','case']).any(), f'{t}: validation log incomplete'
        assert int(ZIP_BOARD.loc[t,'best_epoch'])==int(m['best_epoch'])
        print(t,': ZIP logs complete, best epoch',m['best_epoch'])


In [ ]:
"""Plot recorded HairFastGAN trials; no training or checkpoint loading.
CLI: python plot_train_val.py --source results.zip --out figures
Also accepts an extracted study directory containing trial_A ... trial_D.
"""
from pathlib import Path
import argparse
import json
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

COLORS = dict(A='#3178C6', B='#098C79', C='#DB9031', D='#9966B3')
SPECS = [
 ('loss', 'Total loss ↓'),
 ('L_quantile', 'Lightness error L ↓'),
 ('ab_quantile', 'Color error ab ↓'),
 ('identity_cosine', 'Face identity cosine ↑'),
 ('protected_RGB_MAE_vs_input', 'Protected-region MAE ↓'),
 ('detail_512_MAE_vs_pre', 'Detail MAE vs pre-output ↓'),
 ('color_loss', 'Color loss ↓'),
 ('shape_loss', 'Shape loss ↓'),
 ('target_mask_IoU', 'Hair-mask IoU ↑'),
]
COMPARE = [SPECS[1], SPECS[2], SPECS[8], SPECS[3], SPECS[4], ('score', 'Combined color / shape score ↓')]

class Source:
    def __init__(self, path):
        self.path = Path(path)
        self.z = zipfile.ZipFile(self.path) if self.path.is_file() else None
    def read_csv(self, name):
        if self.z:
            matches = [n for n in self.z.namelist() if n == name or n.endswith('/' + name)]
            if len(matches) != 1:
                raise ValueError(f'Expected one {name}; found {matches}')
            with self.z.open(matches[0]) as f:
                return pd.read_csv(f)
        return pd.read_csv(self.path / name)
    def close(self):
        if self.z: self.z.close()

def style(ax, title):
    ax.set_title(title, loc='left', fontsize=12, fontweight='bold', pad=10)
    ax.grid(alpha=.2)
    ax.spines[['top', 'right']].set_visible(False)
    ax.set_xlabel('Epoch')
    ax.set_xlim(0, 30)
    ax.set_xticks(range(0, 31, 5))

def run(source, out='figures', show=True):
    out = Path(out); out.mkdir(parents=True, exist_ok=True)
    src = Source(source)
    plt.rcParams.update({'font.family':'DejaVu Sans', 'font.size':10, 'figure.facecolor':'white'})
    histories = {}; trains = {}; selections = {}; counts = {}
    try:
        leaderboard = src.read_csv('leaderboard.csv').set_index('trial')
        for t in COLORS:
            steps = src.read_csv(f'trial_{t}/training_steps.csv')
            records = src.read_csv(f'trial_{t}/metrics.csv')
            # Exclude the old comparator (negative epochs), never use its results.
            records = records.loc[(records['split'] == 'validation') & (records.epoch >= 0)].copy()
            steps = steps.loc[steps.epoch >= 1].copy()
            if records.duplicated(['epoch', 'case']).any() or steps['update'].duplicated().any():
                raise ValueError(f'{t}: duplicated records; resolve resumed-run logs first')
            val = records.groupby('epoch').mean(numeric_only=True)
            train = steps.groupby('epoch').mean(numeric_only=True)
            if val.index.tolist() != list(range(31)) or train.index.tolist() != list(range(1,31)):
                raise ValueError(f'{t}: expected validation 0..30 and training 1..30')
            sizes = records.groupby('epoch').size()
            if sizes.nunique() != 1:
                raise ValueError(f'{t}: validation count changes between epochs')
            counts[t] = int(sizes.iloc[0])
            selections[t] = int(leaderboard.loc[t, 'best_epoch'])
            histories[t], trains[t] = val, train
            train.to_csv(out / f'trial_{t}_train_epoch_means.csv')
            val.to_csv(out / f'trial_{t}_validation_epoch_means.csv')
    finally:
        src.close()
    base = histories['A'].loc[0]
    for t,val in histories.items():
        metric_keys=[k for k,_ in SPECS]
        np.testing.assert_allclose(val.loc[0,metric_keys],base[metric_keys],rtol=1e-7,atol=1e-9)
        # Same normalized score used in the supplied leaderboard. Baseline = 1.
        val['score'] = .5*((val.L_quantile+val.ab_quantile)/(base.L_quantile+base.ab_quantile)
                          +(1-val.target_mask_IoU)/(1-base.target_mask_IoU))
        np.testing.assert_allclose(val.loc[selections[t],'score'],leaderboard.loc[t,'score'],rtol=1e-7)
    figures = []
    def save(fig, name, pdf):
        fig.savefig(out / (name+'.png'),dpi=180,bbox_inches='tight')
        pdf.savefig(fig,bbox_inches='tight')
        figures.append(out / (name+'.png'))
        if show: plt.show()
        plt.close(fig)
    with PdfPages(out/'all_train_val_and_comparisons.pdf') as pdf:
        # Display A, then B, C, D before any combined figure.
        for t in COLORS:
            train, val = trains[t], histories[t]; best = selections[t]
            fig, axs = plt.subplots(3,3,figsize=(15,12))
            for ax,(key,title) in zip(axs.flat,SPECS):
                if key in train:
                    ax.plot(train.index,train[key],color='#3178C6',lw=1.8,label='Train: mean of update logs')
                else:
                    ax.text(.04,.92,'Train metric not logged',transform=ax.transAxes,va='top',fontsize=9,color='#667085')
                ax.plot(val.index,val[key],color='#D87523',lw=1.8,label='Validation: end of epoch')
                ax.axhline(base[key],ls=':',color='#7B8494',lw=1.2,label='Pretrained validation baseline')
                ax.axvline(best,ls='--',color='#818898',lw=1,alpha=.6)
                ax.scatter(best,val.loc[best,key],marker='*',s=130,color='#D87523',edgecolor='white',zorder=5)
                style(ax,title)
            fig.suptitle(f'Trial {t} | Train and validation | Selected epoch {best}',fontsize=20,fontweight='bold',y=.985)
            fig.text(.07,.948,f'Blending LR = {leaderboard.loc[t,"blend_lr"]:g}    PostProcess LR = {leaderboard.loc[t,"pp_lr"]:g}    Validation: {counts[t]} pairs',fontsize=11)
            h,l=axs.flat[0].get_legend_handles_labels()
            fig.legend(h,l,loc='lower center',bbox_to_anchor=(.5,.052),ncol=3,frameon=False)
            fig.text(.07,.02,'Train values are recorded during weight updates; validation uses the end-of-epoch model.\nTheir gap is descriptive, not a controlled train-set vs validation-set evaluation. Star = selected checkpoint.',fontsize=10,color='#667085')
            fig.subplots_adjust(top=.89,bottom=.13,hspace=.4,wspace=.29)
            save(fig,f'01_trial_{t}_train_validation',pdf)
        # Combined train/validation: separate columns; common y-scale per metric.
        fig,axs=plt.subplots(3,2,figsize=(13,12),sharex=True)
        for i,(key,title) in enumerate([SPECS[0],SPECS[1],SPECS[2]]):
            limits=[]
            for j,(collection,label) in enumerate([(trains,'Train: update-log means'),(histories,'Validation: end of epoch')]):
                ax=axs[i,j]
                for t,d in collection.items(): ax.plot(d.index,d[key],color=COLORS[t],label=t,lw=1.7)
                style(ax,f'{title} | {label}')
                limits.extend(ax.get_ylim())
            for ax in axs[i]: ax.set_ylim(min(limits),max(limits))
        fig.suptitle('All trials | Train and validation in separate panels',fontsize=19,fontweight='bold')
        fig.legend(*axs[0,0].get_legend_handles_labels(),loc='lower center',ncol=4,frameon=False)
        fig.tight_layout(rect=(0,.04,1,.95));save(fig,'02_all_trials_train_validation',pdf)
        # Same validation comparison layout as the earlier report.
        fig,axs=plt.subplots(2,3,figsize=(15,9))
        for ax,(key,title) in zip(axs.flat,COMPARE):
            ax.axhline(histories['A'].loc[0,key],ls='--',color='#7B8494',label='Baseline')
            for t,val in histories.items():
                ax.plot(val.index,val[key],color=COLORS[t],label=t,lw=1.7)
                e=selections[t];ax.scatter(e,val.loc[e,key],marker='*',s=120,color=COLORS[t],edgecolor='white',zorder=5)
            style(ax,title)
        fig.suptitle('All trials | Validation learning curves',fontsize=20,fontweight='bold')
        fig.legend(*axs.flat[0].get_legend_handles_labels(),loc='lower center',ncol=5,frameon=False)
        fig.tight_layout(rect=(0,.06,1,.95));save(fig,'03_all_trials_validation',pdf)
        rows=[dict(model='Baseline',epoch=0,**histories['A'].loc[0].to_dict())]
        for t,val in histories.items():rows.append(dict(model=t,epoch=selections[t],**val.loc[selections[t]].to_dict()))
        result=pd.DataFrame(rows)
        result.to_csv(out/'best_checkpoints_vs_baseline.csv',index=False)
        fig,axs=plt.subplots(2,3,figsize=(15,9))
        for ax,(key,title) in zip(axs.flat,COMPARE):
            values=result[key].to_numpy()
            ax.bar(range(5),values,color=['#7B8494']+list(COLORS.values()),width=.65)
            ax.set_xticks(range(5),['Baseline']+[f'{t}\nep {selections[t]}' for t in COLORS])
            ax.set_ylim(0,max(values)*1.2)
            ax.set_title(title,loc='left',fontsize=12,fontweight='bold');ax.grid(axis='y',alpha=.18);ax.set_axisbelow(True)
            ax.spines[['top','right']].set_visible(False)
            for i,v in enumerate(values):ax.text(i,v+max(values)*.025,f'{v:.4f}',ha='center',fontsize=10)
        fig.suptitle('Official pretrained baseline vs best checkpoint of A–D',fontsize=19,fontweight='bold')
        fig.text(.06,.025,f'Validation means ({counts["A"]} pairs); not test results. Baseline = epoch 0. Negative-epoch comparator rows excluded.',color='#667085')
        fig.tight_layout(rect=(0,.07,1,.95));save(fig,'04_baseline_vs_best_trials',pdf)
    notes = {
        'source':str(source), 'selected_epochs':selections, 'validation_pairs':counts,
        'train_aggregation':'Unweighted arithmetic mean of update logs per epoch (batch=1 in this study).',
        'validation_aggregation':'Arithmetic mean across validation cases at each epoch.',
        'missing_train_metrics':['target_mask_IoU'],
        'score_formula':'0.5*((L+ab)/(baseline_L+baseline_ab)+(1-IoU)/(1-baseline_IoU))',
        'selection':'Uses existing leaderboard best_epoch, not separate best epoch for each metric.',
        'excluded':'All validation rows with epoch < 0 and all old comparator columns.',
        'caution':'Single-seed validation results; no test evaluation or confidence intervals inferred.'
    }
    (out/'plot_notes.json').write_text(json.dumps(notes,indent=2),encoding='utf-8')
    print('Saved figures and tables to:',out.resolve())
    return result, figures



In [ ]:
history_comparison,history_figures=run(REVIEW_ZIP,LOCAL_REPORT/'01_train_val',show=True)
print('Train/validation plots ready:',LOCAL_REPORT/'01_train_val')


## ประเมินโมเดลใหม่ด้วยไฟล์ใน Drive
ส่วนนี้ต้องใช้ GPU และไฟล์ checkpoint/cache เดิม ไม่ได้อ่านสถานะ running จาก Drive มาใช้

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('กรุณาเลือก Runtime > Change runtime type > T4 GPU แล้ว Run all อีกครั้ง')
print('GPU:',torch.cuda.get_device_name(0))


In [ ]:
%pip install -q opencv-python matplotlib scipy scikit-image scikit-learn pandas tqdm dlib face-alignment gdown addict wandb torchmetrics dill ninja transformers diffusers accelerate fastapi uvicorn python-multipart pyngrok nest-asyncio huggingface_hub
%pip install -q git+https://github.com/openai/CLIP.git
import torch
assert torch.cuda.is_available(), 'เลือก Runtime > Change runtime type > GPU ก่อนรัน'
print('GPU:', torch.cuda.get_device_name(0), '| torch:', torch.__version__)


In [ ]:
from google.colab import drive
from pathlib import Path
import os, shutil, subprocess, sys, json, time
from huggingface_hub import snapshot_download

drive.mount('/content/drive')
LOCAL_REPO = '/content/HairFastGAN'
DRIVE_CACHE_DIR = Path('/content/drive/MyDrive/HairFastGAN_cache/pretrained_models')
RESULTS_ROOT = Path('/content/drive/MyDrive/HairFastGAN_prompt_study')
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
# ใส่ commit ที่บันทึกใน metadata.json เมื่อต้องการทำซ้ำภายหลัง
HAIRFAST_REVISION = '49e98019c9243d1991c19d45764fc3340a498de2'
if not Path(LOCAL_REPO, '.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/AIRI-Institute/HairFastGAN.git', LOCAL_REPO], check=True)
if HAIRFAST_REVISION:
    subprocess.run(['git', '-C', LOCAL_REPO, 'checkout', HAIRFAST_REVISION], check=True)
os.chdir(LOCAL_REPO)
local_models = Path(LOCAL_REPO, 'pretrained_models')
if not local_models.exists() or not any(local_models.iterdir()):
    if DRIVE_CACHE_DIR.exists() and any(DRIVE_CACHE_DIR.iterdir()):
        shutil.copytree(DRIVE_CACHE_DIR, local_models, dirs_exist_ok=True)
    else:
        snapshot = snapshot_download('AIRI-Institute/HairFastGAN', allow_patterns=['pretrained_models/**'])
        shutil.copytree(Path(snapshot, 'pretrained_models'), local_models, dirs_exist_ok=True)
        shutil.copytree(local_models, DRIVE_CACHE_DIR, dirs_exist_ok=True)
# ตรวจ cache สำคัญ รวมถึงกรณีได้ Git-LFS pointer แทน weights
for relative in ['StyleGAN/ffhq.pt', 'encoder4editing/e4e_ffhq_encode.pt']:
    p = local_models / relative
    if not p.exists() or p.stat().st_size < 1000000:
        raise RuntimeError(f'โมเดลขาดหรือไม่สมบูรณ์: {p} โปรดตรวจ cache ก่อนดำเนินการ')
Path('input').mkdir(exist_ok=True)
Path('output').mkdir(exist_ok=True)
REPO_COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Repo commit:', REPO_COMMIT)


In [ ]:
# Compatibility backend: native torch operators also execute on CUDA tensors.
# No PyTorch/CUDA version downgrade; no changes to weights or transfer parameters.
import ast, hashlib, importlib, sys
from pathlib import Path
STYLEGAN_OPS_BACKEND = 'torch_native'

def _native_stylegan_source(source, kind):
    marker = '# HAIRFAST_TORCH_NATIVE_V1'
    if marker in source:
        return source
    tree = ast.parse(source)
    extension = 'fused' if kind == 'fused_act' else 'upfirdn2d_op'
    entry = 'fused_leaky_relu' if kind == 'fused_act' else 'upfirdn2d'
    edits = []
    found_load = found_entry = False
    for node in tree.body:
        if isinstance(node, ast.Assign) and any(isinstance(t, ast.Name) and t.id == extension for t in node.targets):
            if not isinstance(node.value, ast.Call) or not isinstance(node.value.func, ast.Name) or node.value.func.id != 'load':
                raise RuntimeError('รูปแบบ extension เปลี่ยนไป: ' + kind)
            edits.append((node.lineno-1, node.end_lineno, extension + ' = None\n'))
            found_load = True
        if isinstance(node, ast.FunctionDef) and node.name == entry:
            edits.append((node.lineno-1, node.end_lineno, FUSED_NATIVE if kind == 'fused_act' else UP_NATIVE))
            found_entry = True
    if not found_load or not found_entry:
        raise RuntimeError('ไม่พบจุด patch ที่คาดไว้ใน ' + kind)
    lines = source.splitlines(True)
    for start,end,text in sorted(edits, reverse=True):
        lines[start:end] = [text]
    updated = marker + '\n' + ''.join(lines)
    if kind == 'upfirdn2d' and not any(isinstance(n, ast.FunctionDef) and n.name == 'upfirdn2d_native' for n in tree.body):
        updated += '\n\n' + UP_NATIVE_IMPL + '\n'
    compile(updated, kind + '.py', 'exec')
    return updated

FUSED_NATIVE = 'def fused_leaky_relu(input, bias=None, negative_slope=0.2, scale=2 ** 0.5):\n    if bias is not None:\n        input = input + bias.reshape(1, -1, *([1] * (input.ndim - 2)))\n    return torch.nn.functional.leaky_relu(input, negative_slope=negative_slope) * scale\n'
UP_NATIVE = 'def upfirdn2d(input, kernel, up=1, down=1, pad=(0, 0)):\n    return upfirdn2d_native(input, kernel, up, up, down, down,\n                           pad[0], pad[1], pad[0], pad[1])\n'
UP_NATIVE_IMPL = 'def upfirdn2d_native(\n    input, kernel, up_x, up_y, down_x, down_y, pad_x0, pad_x1, pad_y0, pad_y1\n):\n    _, channel, in_h, in_w = input.shape\n    input = input.reshape(-1, in_h, in_w, 1)\n\n    _, in_h, in_w, minor = input.shape\n    kernel_h, kernel_w = kernel.shape\n\n    out = input.view(-1, in_h, 1, in_w, 1, minor)\n    out = F.pad(out, [0, 0, 0, up_x - 1, 0, 0, 0, up_y - 1])\n    out = out.view(-1, in_h * up_y, in_w * up_x, minor)\n\n    out = F.pad(\n        out, [0, 0, max(pad_x0, 0), max(pad_x1, 0), max(pad_y0, 0), max(pad_y1, 0)]\n    )\n    out = out[\n        :,\n        max(-pad_y0, 0) : out.shape[1] - max(-pad_y1, 0),\n        max(-pad_x0, 0) : out.shape[2] - max(-pad_x1, 0),\n        :,\n    ]\n\n    out = out.permute(0, 3, 1, 2)\n    out = out.reshape(\n        [-1, 1, in_h * up_y + pad_y0 + pad_y1, in_w * up_x + pad_x0 + pad_x1]\n    )\n    w = torch.flip(kernel, [0, 1]).view(1, 1, kernel_h, kernel_w)\n    out = F.conv2d(out, w)\n    out = out.reshape(\n        -1,\n        minor,\n        in_h * up_y + pad_y0 + pad_y1 - kernel_h + 1,\n        in_w * up_x + pad_x0 + pad_x1 - kernel_w + 1,\n    )\n    out = out.permute(0, 2, 3, 1)\n    out = out[:, ::down_y, ::down_x, :]\n\n    out_h = (in_h * up_y + pad_y0 + pad_y1 - kernel_h) // down_y + 1\n    out_w = (in_w * up_x + pad_x0 + pad_x1 - kernel_w) // down_x + 1\n\n    return out.view(-1, channel, out_h, out_w)'

def install_native_stylegan_ops(repo):
    root = Path(repo).resolve()
    pairs = sorted((root / 'models').rglob('op/fused_act.py'))
    if not pairs:
        raise RuntimeError('ไม่พบ StyleGAN operator ใน ' + str(root))
    planned = []
    for fused_path in pairs:
        for path,kind in [(fused_path,'fused_act'), (fused_path.with_name('upfirdn2d.py'),'upfirdn2d')]:
            if not path.is_file():
                raise RuntimeError('ขาด operator: ' + str(path))
            module_name = '.'.join(path.relative_to(root).with_suffix('').parts)
            if module_name in sys.modules:
                raise RuntimeError('StyleGAN ถูก import แล้ว: Restart runtime แล้วรันตามลำดับใหม่')
            old = path.read_text()
            planned.append((path, old, _native_stylegan_source(old,kind)))
    # Validate every candidate before writing; preserve each original once.
    records = []
    for path,old,new in planned:
        if old != new:
            backup = path.with_suffix('.py.before_native')
            if not backup.exists(): backup.write_text(old)
            path.write_text(new)
        records.append({'path':str(path.relative_to(root)),
                        'sha256':hashlib.sha256(new.encode()).hexdigest()})
    importlib.invalidate_caches()
    return records

STYLEGAN_NATIVE_PATCHES = install_native_stylegan_ops(LOCAL_REPO)
print('StyleGAN backend:', STYLEGAN_OPS_BACKEND, '| patched files:', len(STYLEGAN_NATIVE_PATCHES))

# Small on-device smoke check before loading model weights.
import torch
_check = {'torch':torch, 'F':torch.nn.functional}
exec(FUSED_NATIVE + '\n' + UP_NATIVE_IMPL + '\n' + UP_NATIVE, _check)
_device = 'cuda' if torch.cuda.is_available() else 'cpu'
_x = torch.randn(1, 2, 5, 5, device=_device, requires_grad=True)
_b = torch.randn(2, device=_device, requires_grad=True)
_y = _check['fused_leaky_relu'](_x, _b, .1)
torch.testing.assert_close(_y, torch.nn.functional.leaky_relu(_x + _b.view(1,2,1,1), .1) * (2 ** .5))
_z = _check['upfirdn2d'](_x, torch.ones(1,1,device=_device))
torch.testing.assert_close(_z, _x)
(_y.sum() + _z.sum()).backward()
assert torch.isfinite(_x.grad).all() and torch.isfinite(_b.grad).all()
print('Native operator smoke check passed on', _device)
del _check, _x, _b, _y, _z


In [ ]:
from PIL import Image
if not hasattr(Image,"ANTIALIAS"): Image.ANTIALIAS=Image.Resampling.LANCZOS
import sys, torch
sys.path.insert(0, LOCAL_REPO)
from hair_swap import HairFast, get_parser
hair_fast = HairFast(get_parser().parse_args([]))
print('HairFastGAN พร้อม')


In [ ]:
import io, csv, zipfile, hashlib, uuid, random, copy, traceback, time, gc
import numpy as np
import pandas as pd
import torch.nn.functional as F
from PIL import Image, ImageDraw
from IPython.display import display
from models.Net import get_segmentation as segment
from torch.utils.checkpoint import checkpoint

# แก้เพียง path นี้หากย้ายโฟลเดอร์ผลการทดลอง
STUDY_ROOT = Path('/content/drive/MyDrive/HairFastGAN_BPP_4Trials_PretrainedStart/9be83b9c2e')
OUT = LOCAL_REPORT/'02_reevaluation'
def sha(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for chunk in iter(lambda:f.read(8*1024*1024), b''): h.update(chunk)
    return h.hexdigest()
MANIFESTS=ZIP_MANIFESTS  # ZIP is the authoritative history; do not read Drive status
REFERENCE_META=MANIFESTS['A']
SOURCE=Path(REFERENCE_META['source'])
VAL_KEYS=REFERENCE_META['config']['validation_keys']
assert len(VAL_KEYS)==9 and len(set(VAL_KEYS))==9
bykey={k:{'split':'validation'} for k in VAL_KEYS}
for t,m in MANIFESTS.items():
    assert m['config']['validation_keys']==VAL_KEYS
    assert m['repo_commit']==REPO_COMMIT
    assert (STUDY_ROOT/f'trial_{t}/best.pt').is_file()
    for k in VAL_KEYS:
        assert sha(SOURCE/'cache'/f'{k}.pt')==m['source_cache_hashes'][k], f'Cache changed: {t}/{k}'
        for mask in ['eval_core','eval_refmask','target','protected']:
            a=STUDY_ROOT/'trial_A/inputs'/k/(mask+'.png')
            other=STUDY_ROOT/f'trial_{t}/inputs'/k/(mask+'.png')
            assert sha(a)==sha(other), f'Masks differ: {t}/{k}/{mask}'
b=hair_fast.blend;enc=b.blending_encoder;G=b.net.generator;pp=b.post_process;align=hair_fast.align
assert sha(b.opts.blending_checkpoint)==REFERENCE_META['init_blending_sha256']
assert sha(b.opts.pp_checkpoint)==REFERENCE_META['init_postprocess_sha256']
for module in [enc,G,pp,align]: module.eval().requires_grad_(False)
WEIGHTS=REFERENCE_META['config']['weights']
OUT.mkdir(parents=True,exist_ok=False)
print('Validation cases:',len(VAL_KEYS),'| New output:',OUT)

# Verify every checkpoint before expensive evaluation; never modify source files.
for t in 'ABCD':
    checkpoint_path=STUDY_ROOT/f'trial_{t}/best.pt'
    ck=torch.load(checkpoint_path,map_location='cpu',weights_only=True)
    assert ck['format']=='hairfast_bpp_pretrained_start_trial_v1', f'{t}: format mismatch'
    assert ck['config']['trial']==t, f'{t}: wrong trial checkpoint'
    assert int(ck['epoch'])==int(MANIFESTS[t]['best_epoch']), f'{t}: checkpoint epoch differs from ZIP; select the matching study directory'
    for field in ['blend_lr','pp_lr','seed','weights','train_keys','validation_keys']:
        assert ck['config'][field]==MANIFESTS[t]['config'][field], f'{t}: config mismatch: {field}'
    assert ck['source_manifest_sha256']==sha(SOURCE/'manifest.json')
    with zipfile.ZipFile(REVIEW_ZIP) as z:
        zip_study=json.loads(z.read(zip_member(z,'study.json')))
    assert ck['mask_zip_sha256']==zip_study['mask_zip_sha256']
    del ck
print('Checkpoint checks passed; no training will run')


In [ ]:
runtime_baseS={}
def cpu(x):
    if torch.is_tensor(x):
        return x.detach().cpu().clone()
    if isinstance(x, dict):
        return {k: cpu(v) for k, v in x.items()}
    if isinstance(x, list):
        return [cpu(v) for v in x]
    if isinstance(x, tuple):
        return tuple((cpu(v) for v in x))
    return copy.deepcopy(x)

def gpu(x):
    if torch.is_tensor(x):
        return x.to('cuda').clone()
    if isinstance(x, dict):
        return {k: gpu(v) for k, v in x.items()}
    if isinstance(x, list):return [gpu(v) for v in x]
    if isinstance(x, tuple):return tuple(gpu(v) for v in x)
    return x

def resize(x):
    return F.interpolate(x, size=(256, 256), mode='bilinear', align_corners=False)

def saveim(x, p):
    a = x.detach().float().cpu()[0].clamp(0, 1).permute(1, 2, 0).numpy()
    Image.fromarray((a * 255).round().astype('uint8')).save(p)

def lab(x):
    x = x.float().clamp(0, 1)
    lin = torch.where(x <= 0.04045, x / 12.92, ((x + 0.055) / 1.055).pow(2.4))
    M = x.new_tensor([[0.4124564, 0.3575761, 0.1804375], [0.2126729, 0.7151522, 0.072175], [0.0193339, 0.119192, 0.9503041]])
    xyz = torch.einsum('ij,bjhw->bihw', M, lin) / x.new_tensor([0.95047, 1.0, 1.08883])[None, :, None, None]
    f = torch.where(xyz > 0.008856, xyz.clamp_min(1e-08).pow(1 / 3), 7.787 * xyz + 16 / 116)
    return torch.stack([116 * f[:, 1] - 16, 500 * (f[:, 0] - f[:, 1]), 200 * (f[:, 1] - f[:, 2])], 1)

def render(s, d):
    S = torch.cat([d['S'][:, :6], s], 1)
    return G([S], input_is_latent=True, return_latents=False, start_layer=4, end_layer=8, layer_in=d['F'], randomize_noise=False)[0]

def finish(raw, d):
    sf, ff = pp(d['I1'], b.downsample_256(raw))
    return G([sf], input_is_latent=True, return_latents=False, start_layer=5, end_layer=8, layer_in=ff, randomize_noise=False)[0]

def detail(x):
    y = x.mean(1, keepdim=True)
    mu = F.avg_pool2d(y, 9, 1, 4)
    v = F.avg_pool2d((y - mu).square(), 9, 1, 4)
    return (y - mu) / (v + 0.001).sqrt()

def unit(raw):return resize(((raw+1)/2).clamp(0,1))
def read_mask(p):
    with Image.open(p) as im:
        if im.size!=(256,256):raise ValueError('Mask size mismatch')
        a=np.asarray(im.convert('L'))>127
    return torch.from_numpy(a.copy())[None,None]

def load_case(k):
    d=torch.load(SOURCE/'cache'/(k+'.pt'),map_location='cpu',weights_only=True)
    folder=STUDY_ROOT/'trial_A/inputs'/k
    for name in ['eval_core','eval_refmask','target','protected']:
        mask=read_mask(folder/(name+'.png'))
        if name in ['target','protected']:
            assert torch.equal(d[name].bool(),mask), (k,name)
        d[name]=mask
    for name in ['core','refmask']:d[name]=d[name].bool()
    for name in ['eval_core','eval_refmask']:assert d[name].sum()>=100
    return d

def quantile_color(im,ref,mask,refmask):
    l=lab(im)[0,:,mask[0,0].bool()];r=lab(ref)[0,:,refmask[0,0].bool()]
    q=torch.linspace(.1,.9,9,device=im.device)
    diff=(torch.quantile(l,q,dim=1)-torch.quantile(r,q,dim=1)).abs()
    return diff[:,0].mean(),diff[:,1:].mean()

def masked_mae(a,b,m):return ((a-b).abs()*m.float()).sum()/(a.shape[1]*m.sum().clamp_min(1))

def shape_metrics(im,d):
    labels=segment(align.to_bisenet(im))
    pred=F.interpolate((labels==13).float().reshape(1,1,*labels.shape[-2:]),(256,256),mode='nearest')>.5
    target=d['target']
    return dict(target_mask_IoU=float((pred&target).sum()/(pred|target).sum().clamp_min(1)),
                target_missing_fraction=float((target&~pred).sum()/target.sum().clamp_min(1)))


# Frozen differentiable evaluators: reuse exact parser weights, load original ArcFace.
from models.CtrlHair.external_code.face_parsing.my_parsing_util import FaceParsing, FaceParsing_tensor
from losses.pp_losses import Backbone
from huggingface_hub import hf_hub_download
id_path=Path('pretrained_models/ArcFace/ir_se50.pth')
if not id_path.is_file():
    downloaded=hf_hub_download(repo_id='AIRI-Institute/HairFastGAN',filename=str(id_path))
    id_path.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(downloaded,id_path)
id_net=Backbone(input_size=112,num_layers=50,drop_ratio=.6,mode='ir_se')
id_net.load_state_dict(torch.load(id_path,map_location='cpu',weights_only=True),strict=True)
id_net=id_net.cuda().eval().requires_grad_(False)
FaceParsing_tensor.parsing_img()
parser_net=FaceParsing.bise_net
if parser_net is None:raise RuntimeError('Original BiSeNet is not initialized')
parser_net.eval().requires_grad_(False)
assert FaceParsing.label_list[17]=='hair' # raw parser label=17; remapped target label=13

def id_features(im):
    x=F.interpolate(im,size=(256,256),mode='bilinear',align_corners=False)*2-1
    x=F.adaptive_avg_pool2d(x[:,:,35:223,32:220],(112,112))
    feats=id_net(x)
    return F.normalize(feats,dim=1)

def constraint_terms(im,d,fx):
    # Do NOT call get_segmentation: its no_grad + argmax would sever training gradients.
    logits=checkpoint(lambda x:parser_net(align.to_bisenet(x))[0],im,use_reentrant=False) if torch.is_grad_enabled() else parser_net(align.to_bisenet(im))[0]
    other=torch.cat((logits[:,:17],logits[:,18:]),1)
    binary=logits[:,17:18]-torch.logsumexp(other,dim=1,keepdim=True)
    target=F.interpolate(d['target'].float(),size=binary.shape[-2:],mode='nearest')
    if target.sum()==0 or (1-target).sum()==0:raise RuntimeError('Degenerate shape target')
    prob=binary.sigmoid()
    bce=F.binary_cross_entropy_with_logits(binary,target,reduction='none')
    balanced=.5*((bce*target).sum()/target.sum()+(bce*(1-target)).sum()/(1-target).sum())
    dice=1-(2*(prob*target).sum()+1)/(prob.sum()+target.sum()+1)
    embedding=checkpoint(id_features,im,use_reentrant=False) if torch.is_grad_enabled() else id_features(im)
    identity=(1-(embedding*fx['face_embedding']).sum(1)).mean()
    return identity,balanced+dice,balanced,dice

def to_size(x,size):
    return F.interpolate(x,size=(size,size),mode='bilinear',align_corners=False)

def detail_terms(full,d,fx):
    values={}
    for size in [256,512]:
        x=to_size(full,size)
        teacher=fx['pre'] if size==256 else fx['pre_512']
        mask=F.interpolate(d['eval_core'].float(),size=(size,size),mode='nearest')
        inner=-F.max_pool2d(-mask,9,1,4)
        if inner.sum()<64:raise RuntimeError('Insufficient interior hair pixels at '+str(size))
        values[size]=masked_mae(detail(x),detail(teacher),inner)
    return values


from scipy.ndimage import binary_erosion,binary_dilation
from skimage.metrics import structural_similarity
_basic_shape_metrics=shape_metrics
def shape_metrics(im,d):
    result=_basic_shape_metrics(im,d)
    labels=segment(align.to_bisenet(im));p=F.interpolate((labels==13).float().reshape(1,1,*labels.shape[-2:]),(256,256),mode='nearest')[0,0].cpu().numpy()>.5
    t=d['target'][0,0].cpu().numpy()>.5
    result['target_Dice']=float(2*(p&t).sum()/max(p.sum()+t.sum(),1))
    a=p^binary_erosion(p);bnd=t^binary_erosion(t)
    prec=(a&binary_dilation(bnd,iterations=2)).sum()/max(a.sum(),1)
    rec=(bnd&binary_dilation(a,iterations=2)).sum()/max(bnd.sum(),1)
    result['boundary_F1_2px']=float(2*prec*rec/max(prec+rec,1e-12))
    return result

def extra_metrics(im,d):
    x=im[0].permute(1,2,0).cpu().numpy();y=d['face'][0].permute(1,2,0).cpu().numpy()
    mask=d['protected'][0,0].cpu().numpy()>.5
    _,ss=structural_similarity(x,y,data_range=1.,channel_axis=2,full=True)
    safe=binary_erosion(mask,iterations=3)
    if not safe.any():raise ValueError('No protected SSIM interior')
    mse=float(((x-y)**2)[mask].mean())
    return dict(protected_PSNR=float(-10*np.log10(max(mse,1e-12))),protected_SSIM=float(ss[safe].mean()),
        any_channel_255_fraction=float((im[0,:,d['eval_core'][0,0]].max(0).values>=1-0.5/255).float().mean()))


In [ ]:
def joint_forward(d,grad=False):
    def blend_fn(a,c,x,y):return enc(a,c,x,y)
    args=[d['args'][i] for i in range(4)]
    s=checkpoint(blend_fn,*args,use_reentrant=False) if grad else blend_fn(*args)
    raw_pre=checkpoint(lambda a:render(a,d),s,use_reentrant=False) if grad else render(s,d)
    pp_in=b.downsample_256(raw_pre)
    sf,ff=checkpoint(lambda a,c:pp(a,c),d['I1'],pp_in,use_reentrant=False) if grad else pp(d['I1'],pp_in)
    def generate(a,c):return G([a],input_is_latent=True,return_latents=False,
        start_layer=5,end_layer=8,layer_in=c,randomize_noise=False)[0]
    raw=checkpoint(generate,sf,ff,use_reentrant=False) if grad else generate(sf,ff)
    return raw_pre,raw,s,sf,ff

def objective(raw_pre,raw,s,sf,ff,d,fx):
    full=((raw+1)/2).clamp(0,1);im=to_size(full,256);pre=unit(raw_pre)
    L,ab=quantile_color(im,d['ref'],d['eval_core'],d['eval_refmask'])
    preL,preab=quantile_color(pre,d['ref'],d['eval_core'],d['eval_refmask'])
    identity,shape,_,_=constraint_terms(im,d,fx)
    _,preshape,_,_=constraint_terms(pre,d,fx)
    protected=masked_mae(im,d['face'],d['protected'])
    details=detail_terms(full,d,fx);texture=torch.stack(list(details.values())).mean()
    anchor=(s-fx['s']).square().mean()+(sf-fx['sf']).square().mean()+(ff-fx['ff']).square().mean()
    color=(L+ab)/100;precolor=(preL+preab)/100
    loss=(WEIGHTS['final_color']*color+WEIGHTS['final_shape']*shape+
          WEIGHTS['pre_color']*precolor+WEIGHTS['pre_shape']*preshape+
          WEIGHTS['identity']*identity+WEIGHTS['protected']*protected+
          WEIGHTS['detail']*texture+WEIGHTS['anchor']*anchor)
    return loss,dict(loss=loss,color_loss=color,shape_loss=shape,pre_color_loss=precolor,
        pre_shape_loss=preshape,identity_loss=identity,identity_cosine=1-identity,
        protected_RGB_MAE_vs_input=protected,detail_512_MAE_vs_pre=details[512],
        detail_MAE_vs_pre=details[256],detail_training_term=texture,latent_anchor=anchor,
        L_quantile=L,ab_quantile=ab,pre_L_quantile=preL,pre_ab_quantile=preab)

In [ ]:
def reset_pretrained():
    sd=torch.load(b.opts.blending_checkpoint,map_location='cpu',weights_only=True)['model_state_dict']
    missing,unexpected=enc.load_state_dict(sd,strict=False)
    if unexpected or any(not k.startswith('clip_model.') for k in missing):
        raise RuntimeError('Pretrained Blending schema mismatch')
    pp.load_state_dict(torch.load(b.opts.pp_checkpoint,map_location='cpu',weights_only=True)['model_state_dict'],strict=True)
    enc.eval();pp.eval()

# Compute shared baseline detail/latent references once from original pretrained weights.
random.seed(3407);np.random.seed(3407);torch.manual_seed(3407);torch.cuda.manual_seed_all(3407)
reset_pretrained()
fixed={}
with torch.no_grad():
    for k in VAL_KEYS:
        d=gpu(load_case(k));pre,raw,s,sf,ff=joint_forward(d)
        fixed[k]=cpu(dict(pre=unit(pre),pre_512=to_size(((pre+1)/2).clamp(0,1),512),s=s,sf=sf,ff=ff,face_embedding=id_features(d['face'])))
        del d,pre,raw,s,sf,ff

rows=[];provenance={};epochs={'Baseline':0}
with torch.no_grad():
    for model in ['Baseline','A','B','C','D']:
        reset_pretrained() # Every model starts with an explicit complete reset.
        if model!='Baseline':
            path=STUDY_ROOT/f'trial_{model}/best.pt'
            ck=torch.load(path,map_location='cpu',weights_only=True)
            assert ck['format']=='hairfast_bpp_pretrained_start_trial_v1'
            assert ck['config']['trial']==model
            assert ck['repo_commit']==REPO_COMMIT
            assert ck['config']['validation_keys']==VAL_KEYS
            assert ck['source_manifest_sha256']==sha(SOURCE/'manifest.json')
            assert ck['config']['weights']==WEIGHTS
            assert int(ck['epoch'])==int(MANIFESTS[model]['best_epoch'])
            assert ck['start_blending_sha256']==REFERENCE_META['init_blending_sha256']
            sd=ck['blending_trainable_state_dict']
            expected={n for n,p in enc.named_parameters() if n.startswith('modulation_module_list.')}
            assert set(sd)==expected
            state=enc.state_dict();state.update(sd);enc.load_state_dict(state,strict=True)
            pp.load_state_dict(ck['post_process_state_dict'],strict=True)
            epochs[model]=int(ck['epoch'])
            provenance[model]={'checkpoint':str(path),'sha256':sha(path),'epoch':epochs[model]}
            del ck,state,sd
        else:
            provenance[model]={'blending_sha256':sha(b.opts.blending_checkpoint),'postprocess_sha256':sha(b.opts.pp_checkpoint),'epoch':0}
        for k in VAL_KEYS:
            d=gpu(load_case(k));fx=gpu(fixed[k])
            pre,raw,s,sf,ff=joint_forward(d)
            _,metrics=objective(pre,raw,s,sf,ff,d,fx)
            im=unit(raw)
            metrics.update(shape_metrics(im,d));metrics.update(extra_metrics(im,d))
            vals={key:float(value) for key,value in metrics.items()}
            if not all(np.isfinite(v) for v in vals.values()):raise ValueError((model,k,'Non-finite metrics'))
            rows.append(dict(model=model,epoch=epochs[model],case=k,split='validation',**vals))
            folder=OUT/'images'/k;folder.mkdir(parents=True,exist_ok=True)
            saveim((raw+1)/2,folder/(model+'.png'))
            if model=='Baseline':
                for field in ['face','aligned','ref']:saveim(d[field],folder/(field+'.png'))
            del d,fx,pre,raw,s,sf,ff,metrics,im
            print(model,k,'done',flush=True)
        pd.DataFrame(rows).to_csv(OUT/'per_case_metrics.csv',index=False)
reset_pretrained()
frame=pd.DataFrame(rows)
summary=frame.groupby('model').mean(numeric_only=True).reindex(['Baseline','A','B','C','D'])
base=summary.loc['Baseline']
summary['score']=.5*((summary.L_quantile+summary.ab_quantile)/max(base.L_quantile+base.ab_quantile,1e-8)+(1-summary.target_mask_IoU)/max(1-base.target_mask_IoU,1e-8))
summary.to_csv(OUT/'summary_metrics.csv')
(OUT/'evaluation_manifest.json').write_text(json.dumps(dict(models=provenance,validation_keys=VAL_KEYS,repo_commit=REPO_COMMIT,source=str(SOURCE),evaluation='Fresh inference on validation only; fixed pretrained detail targets',score_formula='0.5 * ((L+ab)/(baseline L+ab) + (1-IoU)/(1-baseline IoU))'),indent=2))
display(summary[['epoch','L_quantile','ab_quantile','target_mask_IoU','identity_cosine','protected_RGB_MAE_vs_input','detail_512_MAE_vs_pre','score']])


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from IPython.display import display
colors=['#7B8494','#3178C6','#098C79','#DB9031','#9966B3']
specs=[('L_quantile','Lightness error L ↓'),('ab_quantile','Color error ab ↓'),('target_mask_IoU','Hair-mask IoU ↑'),('identity_cosine','Face identity cosine ↑'),('protected_RGB_MAE_vs_input','Protected-region MAE ↓'),('detail_512_MAE_vs_pre','Detail MAE vs baseline pre-output ↓'),('boundary_F1_2px','Hair boundary F1 ↑'),('protected_SSIM','Protected-region SSIM ↑'),('score','Combined color / shape score ↓')]
with PdfPages(OUT/'comparison_report.pdf') as pdf:
    # Each image uses the same case for all five models.
    for k in VAL_KEYS:
        fig,axes=plt.subplots(2,4,figsize=(16,9))
        names=[('face','Input face'),('aligned','Cached shape-aligned reference'),('ref','Color reference'),('Baseline','Pretrained baseline')]+[(t,f'Trial {t} | epoch {epochs[t]}') for t in 'ABCD']
        for ax,(name,label) in zip(axes.flat,names):
            with Image.open(OUT/'images'/k/(name+'.png')) as im:ax.imshow(im)
            ax.set_title(label,fontsize=11);ax.axis('off')
        fig.suptitle(k+' | Same cached inputs and masks',fontsize=17)
        fig.tight_layout();fig.savefig(OUT/(k+'_comparison.png'),dpi=160);pdf.savefig(fig)
        plt.show();plt.close(fig)
    fig,axes=plt.subplots(3,3,figsize=(15,12))
    for ax,(key,label) in zip(axes.flat,specs):
        values=summary[key].to_numpy()
        ax.bar(range(5),values,color=colors,width=.65)
        ax.set_xticks(range(5),['Baseline']+[f'{t}\nep {epochs[t]}' for t in 'ABCD'])
        ax.set_ylim(0,max(values)*1.2);ax.set_title(label,loc='left',fontsize=11)
        ax.spines[['top','right']].set_visible(False);ax.grid(axis='y',alpha=.2);ax.set_axisbelow(True)
        for i,v in enumerate(values):ax.text(i,v+max(values)*.025,f'{v:.4f}',ha='center',fontsize=9)
    fig.suptitle('Fresh evaluation | Baseline vs A–D | Validation mean, n=9',fontsize=18)
    fig.tight_layout(rect=(0,0,1,.96));fig.savefig(OUT/'metrics_comparison.png',dpi=180);pdf.savefig(fig)
    plt.show();plt.close(fig)
print('Completed:',OUT)
# This is a checkpoint evaluation, not a new epoch-by-epoch training history.


## Test: ชุดข้อมูล คู่ทดสอบและล็อก checkpoint

In [ ]:
# TEST: no optimizer, no backward, no checkpoint selection from test scores.
TEST_OUT=LOCAL_REPORT/'03_test';TEST_OUT.mkdir(exist_ok=True)
DATASET_ZIP=''  # Optional path to HairFastGAN_Dataset_Split_v2.zip in Drive; blank uploads the dataset.
from pathlib import PurePosixPath
if not DATASET_ZIP:
    print('Upload HairFastGAN_Dataset_Split_v2.zip containing the test images (not the review ZIP)')
    up=files.upload()
    if len(up)!=1:raise ValueError('Select one dataset ZIP')
    DATASET_ZIP=str(Path(next(iter(up))).resolve())
DATASET_ZIP=Path(DATASET_ZIP)
assert sha(DATASET_ZIP)=='5eb3a720c71989e6e36e1a38fae13840ec9e9853b5196893e190db499aa5719a','Dataset ZIP differs from reviewed v2'
DATA_DIR=Path('/content')/('test_data_'+sha(DATASET_ZIP)[:12]);DATA_DIR.mkdir(exist_ok=True)
with zipfile.ZipFile(DATASET_ZIP) as z:
    if sum(i.file_size for i in z.infolist())>600*1024**2:raise ValueError('Unexpected dataset size')
    for info in z.infolist():
        p=PurePosixPath(info.filename)
        if p.is_absolute() or '..' in p.parts or ((info.external_attr>>16)&0o170000)==0o120000:raise ValueError('Unsafe ZIP entry')
    z.extractall(DATA_DIR)
mp=list(DATA_DIR.rglob('manifest.json'))
assert len(mp)==1
DATA_ROOT=mp[0].parent
DATA_META=json.loads(mp[0].read_text());ITEMS=DATA_META['items'];ITEM_BY_ID={r['id']:r for r in ITEMS}
for item in ITEMS:
    p=DATA_ROOT/item['file']
    assert p.resolve().is_relative_to(DATA_ROOT.resolve())
    assert sha(p)==item['sha256'],item['id']
test_items=json.loads((DATA_ROOT/'test_candidate.json').read_text())
assert len(test_items)==20
for item in test_items:assert ITEM_BY_ID[item['id']]['split']=='test_candidate'
trainval_cases=json.loads((SOURCE/'cases.json').read_text())
def resolve_dataset_id(value,role):
    category={'face':'faces','shape':'styles','color':'colors'}[role]
    token=str(value);stem=Path(token).stem
    found=[r['id'] for r in ITEMS if r['category']==category and (r['id']==token or r['id']==stem or r['file']==token or r.get('original_path')==token)]
    if len(found)!=1:raise ValueError(f'Cannot resolve historical {role}: {value}')
    return found[0]
used_ids={resolve_dataset_id(c[role],role) for c in trainval_cases if c['split'] in ['train','validation'] for role in ['face','shape','color']}
used_ids.update(r['id'] for r in ITEMS if r['split'] in ['train','validation'])
# Check across roles: same image bytes, decoded pixels, and declared identity/template groups.
used=[ITEM_BY_ID[k] for k in used_ids]
used_hashes={r['sha256'] for r in used};used_pixels={r.get('pixels') for r in used if r.get('pixels')}
used_groups={r['group_id'] for r in used}
for row in test_items:
    r=ITEM_BY_ID[row['id']]
    assert r['id'] not in used_ids and r['sha256'] not in used_hashes,('overlap',r['id'])
    assert r.get('pixels') not in used_pixels,('pixel overlap',r['id'])
    assert r['group_id'] not in used_groups,('group overlap',r['id'])
faces=sorted([r['id'] for r in test_items if r['category']=='faces'])
styles=sorted([r['id'] for r in test_items if r['category']=='styles'])
colors=sorted([r['id'] for r in test_items if r['category']=='colors'])
assert (len(faces),len(styles),len(colors))==(10,5,5)
rng=random.Random(3407);rng.shuffle(styles);rng.shuffle(colors)
TEST_CASES=[dict(key=f'test_{i:03d}',face=f,shape=styles[i%5],color=colors[i%5],face_group=ITEM_BY_ID[f]['group_id']) for i,f in enumerate(faces)]
(TEST_OUT/'test_pairs.json').write_text(json.dumps(TEST_CASES,indent=2))
TEST_KEYS=[r['key'] for r in TEST_CASES]
# Freeze validation-selected weights before generating any test output.
LOCK_DIR=Path('/content')/('locked_ABCD_'+uuid.uuid4().hex[:8]);LOCK_DIR.mkdir()
LOCK={'Baseline':{'blending_sha256':sha(b.opts.blending_checkpoint),'postprocess_sha256':sha(b.opts.pp_checkpoint),'epoch':0}}
for t in 'ABCD':
    original=STUDY_ROOT/f'trial_{t}/best.pt';snap=LOCK_DIR/f'{t}.pt'
    expected_hash=provenance[t]['sha256']
    assert sha(original)==expected_hash,f'{t}: checkpoint changed since validation evaluation'
    shutil.copy2(original,snap);assert sha(snap)==expected_hash
    ck=torch.load(snap,map_location='cpu',weights_only=True)
    assert int(ck['epoch'])==int(MANIFESTS[t]['best_epoch'])
    LOCK[t]=dict(epoch=int(ck['epoch']),sha256=expected_hash,original=str(original),snapshot=str(snap))
    del ck
(TEST_OUT/'checkpoint_lock.json').write_text(json.dumps(LOCK,indent=2))
TEST_PROTOCOL=dict(split='test_candidate',pairs=TEST_CASES,seed=3407,shape_rotation=False,color_rotation=True,mixing=.95,
    mask_policy='fixed automatic eroded hair masks; not manually reviewed validation masks',
    overlap_check='IDs, SHA256, manifest pixel hashes and declared groups vs current train/validation passed',
    historical_overlap='not verified against every older development/training dataset; candidate test results',
    face_group_count=len({r['face_group'] for r in TEST_CASES}),selection='checkpoints fixed by validation; no test-based selection',
    pairing='10 faces once each; shuffled 5 shapes and 5 colors repeated twice, seed 3407; frozen before outputs',repo_commit=REPO_COMMIT)
(TEST_OUT/'test_protocol.json').write_text(json.dumps(TEST_PROTOCOL,indent=2))
display(pd.DataFrame(TEST_CASES))
print('Candidate test split: 10 cases; older-development overlap remains unverified. No test-based tuning.')



## Test: ฟังก์ชันเตรียมภาพและ masks

In [ ]:
# Frozen preprocessing. Reuse official SEAN decode/e4e; do not use the custom trainable decoder.
from utils.shape_predictor import align_face
from models.encoder4editing.utils.model_utils import get_latents
from models.sean_codes.models.pix2pix_model import encode_sean,decode_sean
from models.Alignment import get_hair_face_code,get_new_shape
TEST_CACHE=Path('/content')/('test_cache_'+uuid.uuid4().hex[:8]);TEST_CACHE.mkdir()
e4e=hair_fast.embed.e4e
def erode_test(m,k=7):return -F.max_pool2d(-m.float(),k,1,k//2)
def dilate_test(m,k=11):return F.max_pool2d(m.float(),k,1,k//2)
@torch.no_grad()
def test_embedding(ident):
    path=TEST_CACHE/(ident+'.pt')
    if path.exists():return gpu(torch.load(path,map_location='cpu',weights_only=True))
    with Image.open(DATA_ROOT/ITEM_BY_ID[ident]['file']) as im:original=im.convert('RGB').copy()
    image=align_face([original])[0].unsqueeze(0).cuda();emb=hair_fast.embed
    i512=emb.downsample_512(image);i256=emb.downsample_256(image)
    W=get_latents(e4e,emb.normalize(i256))
    output=emb.encoder.test(img=emb.normalize(image),return_latent=True)
    feature16=output[-1];S=output[-2]
    feature32=G([S],input_is_latent=True,return_latents=False,start_layer=3,end_layer=3,layer_in=feature16,randomize_noise=False)[0]
    mask=segment(emb.to_bisenet(i512))
    h=F.interpolate((mask==13).float(),(32,32),mode='bicubic')
    fw=G([W],input_is_latent=True,return_latents=False,start_layer=0,end_layer=3,randomize_noise=False)[0]
    feature32=feature32+.95*h*(fw-feature32)
    result=cpu(dict(W=W,F=feature32,S=S,mask=mask,image_256=i256,image_norm_256=emb.normalize(i256)))
    torch.save(result,path);return gpu(result)

@torch.no_grad()
def prepare_test_case(row):
    key=row['key']
    seed=(3407+int(hashlib.sha256(key.encode()).hexdigest()[:8],16))%(2**31)
    torch.manual_seed(seed);torch.cuda.manual_seed_all(seed)
    ef,es,ec=[test_embedding(row[r]) for r in ['face','shape','color']]
    face_code,_=get_hair_face_code(align.mask_generator,ef['mask'][0,0])
    _,hair_code=get_hair_face_code(align.mask_generator,es['mask'][0,0])
    labels=get_new_shape(align.mask_generator,face_code,hair_code)[None,None].long()
    target=labels==13;hf=(ef['mask']==13).float();hs=(es['mask']==13).float()
    codes=encode_sean(align.sean_model,torch.cat([ef['image_256'],es['image_256']],0),torch.cat([ef['mask'],es['mask']],0))
    gen1=decode_sean(align.sean_model,codes[0:1],labels)
    gen2=decode_sean(align.sean_model,codes[1:2],labels)
    aligned_embed=align.latent_encoder([gen1,gen2])
    a=aligned_embed['F'][0:1];c=aligned_embed['F'][1:2]
    masks=torch.cat([1-(1-hf)*(1-target.float()),target.float(),hs*target.float()],0)
    dil,ero=align.dilate_erosion.mask(masks)
    low=1-F.interpolate(torch.stack([dil[0],ero[1],ero[2]],0).float(),(32,32),mode='bicubic')
    feature=a+low[0]*(ef['F']-a);feature=c+low[1]*(feature-c);feature=es['F']+low[2]*(feature-es['F'])
    color_align=align.shape_module('face','color',{'face':ef,'color':ec})
    de=b.dilate_erosion.hair_from_mask(torch.cat([ef['mask'],ec['mask']],0))
    h1d=de[0][0].unsqueeze(0);h3d=de[0][1].unsqueeze(0);h3e=de[1][1].unsqueeze(0)
    hxd,_=b.dilate_erosion.mask(color_align['HM_X'])
    preserve=(1-h1d)*(1-h3d)*(1-hxd)
    protected=dilate_test(hf.bool()|target)<.5
    core=(erode_test(target)>.5)&~protected;refmask=erode_test(ec['mask']==13)>.5
    if min(int(core.sum()),int(refmask.sum()),int(protected.sum()))<100:raise ValueError('Insufficient mask pixels: '+key)
    aligned=G([ef['S']],input_is_latent=True,return_latents=False,start_layer=4,end_layer=8,layer_in=feature,randomize_noise=False)[0]
    d=dict(S=ef['S'],F=feature,I1=ef['image_norm_256'],face=ef['image_256'],shape=es['image_256'],ref=ec['image_256'],aligned=unit(aligned),target=target,protected=protected,core=core,refmask=refmask,eval_core=core,eval_refmask=refmask,
        args=[ef['S'][:,6:],ec['S'][:,6:],ef['image_norm_256']*preserve,ec['image_norm_256']*h3e])
    torch.save(cpu(d),TEST_CACHE/(key+'.pt'))



## Test: เตรียม cache คงที่สำหรับทั้ง 5 โมเดล

In [ ]:
reset_pretrained()
for row in TEST_CASES:
    prepare_test_case(row);print('Prepared test',row['key'])
# A preparation failure stops the run; no silent deletion or replacement of test cases.
TEST_FIXED={}
with torch.no_grad():
    for row in TEST_CASES:
        key=row['key'];d=gpu(torch.load(TEST_CACHE/(key+'.pt'),map_location='cpu',weights_only=True))
        pre,raw,s,sf,ff=joint_forward(d)
        TEST_FIXED[key]=cpu(dict(pre=unit(pre),pre_512=to_size(((pre+1)/2).clamp(0,1),512),s=s,sf=sf,ff=ff,face_embedding=id_features(d['face'])))
        folder=TEST_OUT/'images'/key;folder.mkdir(parents=True,exist_ok=True)
        for name in ['face','shape','ref','aligned']:saveim(d[name],folder/(name+'.png'))
        for name in ['target','protected','eval_core','eval_refmask']:
            Image.fromarray((d[name][0,0].cpu().numpy()*255).astype('uint8')).save(folder/(name+'.png'))
        del d,pre,raw,s,sf,ff


## Test: รัน baseline และ A–D โดยไม่ฝึก

In [ ]:
TEST_ROWS=[]
try:
    with torch.no_grad():
        for model in ['Baseline','A','B','C','D']:
            assert sha(b.opts.blending_checkpoint)==LOCK['Baseline']['blending_sha256']
            assert sha(b.opts.pp_checkpoint)==LOCK['Baseline']['postprocess_sha256']
            reset_pretrained()
            if model!='Baseline':
                path=Path(LOCK[model]['snapshot']);assert sha(path)==LOCK[model]['sha256']
                ck=torch.load(path,map_location='cpu',weights_only=True)
                state=enc.state_dict();state.update(ck['blending_trainable_state_dict']);enc.load_state_dict(state,strict=True)
                pp.load_state_dict(ck['post_process_state_dict'],strict=True);del state,ck
            for row in TEST_CASES:
                key=row['key'];d=gpu(torch.load(TEST_CACHE/(key+'.pt'),map_location='cpu',weights_only=True));fx=gpu(TEST_FIXED[key])
                pre,raw,s,sf,ff=joint_forward(d);_,values=objective(pre,raw,s,sf,ff,d,fx)
                im=unit(raw);values.update(shape_metrics(im,d));values.update(extra_metrics(im,d))
                numeric={k:float(v) for k,v in values.items()}
                assert all(np.isfinite(v) for v in numeric.values()),(model,key)
                TEST_ROWS.append(dict(model=model,epoch=LOCK[model]['epoch'],case=key,split='test_candidate',face_group=row['face_group'],**numeric))
                saveim((raw+1)/2,TEST_OUT/'images'/key/(model+'.png'))
                pd.DataFrame(TEST_ROWS).to_csv(TEST_OUT/'per_case_metrics.csv',index=False)
                del d,fx,pre,raw,s,sf,ff,values,im
                print('Test',model,key,'done')
finally:
    reset_pretrained()


## Test: ตาราง กราฟและภาพเปรียบเทียบ

In [ ]:
TEST_FRAME=pd.DataFrame(TEST_ROWS)
assert len(TEST_FRAME)==5*len(TEST_CASES) and not TEST_FRAME.duplicated(['model','case']).any()
ORDER=['Baseline','A','B','C','D']
TEST_MEAN=TEST_FRAME.groupby('model').mean(numeric_only=True).reindex(ORDER)
TEST_SD=TEST_FRAME.groupby('model').std(numeric_only=True,ddof=1).reindex(ORDER)
tb=TEST_MEAN.loc['Baseline']
TEST_MEAN['score']=.5*((TEST_MEAN.L_quantile+TEST_MEAN.ab_quantile)/max(tb.L_quantile+tb.ab_quantile,1e-8)+(1-TEST_MEAN.target_mask_IoU)/max(1-tb.target_mask_IoU,1e-8))
TEST_MEAN.to_csv(TEST_OUT/'mean_metrics.csv');TEST_SD.to_csv(TEST_OUT/'sd_across_cases.csv')
TEST_FRAME.groupby(['model','face_group']).mean(numeric_only=True).to_csv(TEST_OUT/'mean_by_face_group.csv')
# The score is a ratio of aggregate means; no misleading per-case SD is assigned to it.
cols=['L_quantile','ab_quantile','target_mask_IoU','identity_cosine','protected_RGB_MAE_vs_input','detail_512_MAE_vs_pre']
pd.DataFrame([dict(model=m,metric=k,mean=TEST_MEAN.loc[m,k],sd_cases=TEST_SD.loc[m,k],n_cases=len(TEST_CASES)) for m in ORDER for k in cols]).to_csv(TEST_OUT/'mean_sd_long.csv',index=False)
display(TEST_MEAN[['epoch']+cols+['score']])
print('All five models evaluated. SD describes spread across cases; repeated identities are not independent samples. No test winner is selected.')
from matplotlib.backends.backend_pdf import PdfPages
with PdfPages(TEST_OUT/'test_report.pdf') as pdf:
    for row in TEST_CASES:
        key=row['key'];fig,axs=plt.subplots(2,4,figsize=(16,9))
        panels=[('face','Input'),('shape','Shape reference'),('ref','Color reference'),('Baseline','Baseline')]+[(t,f'{t} | epoch {LOCK[t]["epoch"]}') for t in 'ABCD']
        for ax,(name,label) in zip(axs.flat,panels):
            with Image.open(TEST_OUT/'images'/key/(name+'.png')) as im:ax.imshow(im)
            ax.set_title(label);ax.axis('off')
        fig.suptitle(key+' | fixed test-candidate pair');fig.tight_layout();pdf.savefig(fig);fig.savefig(TEST_OUT/(key+'_comparison.png'),dpi=150);plt.show();plt.close(fig)
    fig,axs=plt.subplots(2,3,figsize=(15,9))
    labels=['Lightness error L ↓','Color error ab ↓','Hair-mask IoU ↑','Face identity cosine ↑','Protected MAE ↓','Detail MAE vs baseline pre-output ↓']
    for ax,k,label in zip(axs.flat,cols,labels):
        y=TEST_MEAN[k].to_numpy();e=TEST_SD[k].to_numpy()
        ax.errorbar(range(5),y,yerr=e,fmt='none',color='#667085',capsize=4)
        ax.scatter(range(5),y,c=['#7B8494','#3178C6','#098C79','#DB9031','#9966B3'],s=70,zorder=4)
        ax.set_xticks(range(5),ORDER);ax.set_title(label,loc='left');ax.grid(alpha=.2)
    fig.suptitle('Test candidate | Mean ± SD across 10 cases (not confidence intervals)')
    fig.tight_layout();pdf.savefig(fig);fig.savefig(TEST_OUT/'test_metrics_mean_sd.png',dpi=180);plt.show();plt.close(fig)
(TEST_OUT/'completion.json').write_text(json.dumps(dict(status='complete',models=ORDER,cases=len(TEST_CASES),metric_rows=len(TEST_ROWS),no_test_based_selection=True),indent=2))
print('Test results:',TEST_OUT)


In [ ]:
assert (LOCAL_REPORT/'03_test/completion.json').is_file(), 'Run TEST cells before exporting the full report'
# Both sections in a single report; historical curves are not relabeled as fresh inference.
keys=['L_quantile','ab_quantile','target_mask_IoU','identity_cosine','protected_RGB_MAE_vs_input','detail_512_MAE_vs_pre','score']
historical=history_comparison.set_index('model')
deltas=[]
for model in ['Baseline','A','B','C','D']:
    assert int(historical.loc[model,'epoch'])==int(epochs[model])
    for metric in keys:
        old=float(historical.loc[model,metric]);new=float(summary.loc[model,metric])
        deltas.append(dict(model=model,metric=metric,history=old,reevaluation=new,delta=new-old))
pd.DataFrame(deltas).to_csv(LOCAL_REPORT/'history_vs_reevaluation.csv',index=False)
(LOCAL_REPORT/'README.txt').write_text('01_train_val: recorded history from the uploaded review ZIP.\n02_reevaluation: fresh inference from matching best checkpoints and pretrained baseline.\nAll metrics use the same validation cases; not test results. No retraining.\n',encoding='utf-8')
(LOCAL_REPORT/'source_info.json').write_text(json.dumps(dict(review_zip=REVIEW_ZIP.name,review_zip_sha256=sha(REVIEW_ZIP),study_root=str(STUDY_ROOT)),indent=2))
archive=shutil.make_archive(str(LOCAL_REPORT),'zip',LOCAL_REPORT)
# Keep one durable archive in Drive; download once.
saved=STUDY_ROOT/Path(archive).name
shutil.copy2(archive,saved)
print('Saved in Drive:',saved)
print('Local report:',LOCAL_REPORT)
from google.colab import files
files.download(archive)
